## Pipeline for deep learning modeling

In [1]:
import os

# Check if it's in the correct directory
print("Current working directory:", os.getcwd())
path = os.path.abspath(os.path.join(os.getcwd(), '..', 'path.py'))
%run $path

Current working directory: c:\Users\vinic\Dropbox\baseline\baseline\notebooks


##### Configure notebook

In [6]:
# Import data
train_file = '../data/mico/scaffold/train.csv'
val_file = '../data/mico/scaffold/val.csv'
test_file = '../data/mico/scaffold/test.csv'

# Configure dataloader
batch_size=128

# Modeling parameters
architecture_type='mpnn'
lr = 0.0005
n_trials=50

# Save the trained model
model_directory = '../output/models'
params_directory = '../output/params'
filename = 'model_mpnn-scaffold'
fig_path1='../output/figures'

# Plot embeddings along the epochs
method='tSNE'
emb_path='../output/embeddings'
fig_path2='../output/figures'
fig_path3='../output/figures'

##### Load data

In [7]:
from params import load_data

train_smiles, y_train = load_data(train_file)
val_smiles, y_val = load_data(val_file)
test_smiles, y_test = load_data(test_file)

print(f"Training data: {len(train_smiles)} samples")
print(f"Validation data: {len(val_smiles)} samples")
print(f"Test data: {len(test_smiles)} samples")

Training data: 14191 samples
Validation data: 1773 samples
Test data: 1775 samples


##### Building molecular graphs in data loaders

In [8]:
from loaders import graph_loader, graph_info

train_loader, val_loader, test_loader = graph_loader(
    train_smiles,
    val_smiles,
    test_smiles,
    y_train,
    y_val,
    y_test,
    batch_size=batch_size,
    seed=42)

node_dim, edge_dim, num_tasks = graph_info(train_loader)
print(f"Max number of atom features: {node_dim}")
print(f"Max number of bond features: {edge_dim}")
print(f"Number of tasks: {num_tasks}")

Max number of atom features: 48
Max number of bond features: 12
Number of tasks: 9


##### Starting optuna optimization

In [9]:
from optimizer import objective
from params import initialize_optuna

study = initialize_optuna()
study.optimize(lambda trial: objective(
    trial,
    node_dim,
    edge_dim,
    train_loader,
    val_loader,
    num_tasks,
    architecture_type=architecture_type,
    lr=lr),
    n_trials=n_trials)

best_params = study.best_params
best_trial = study.best_trial
print("Best hyperparameters:", best_params)

[I 2026-07-25 18:44:59,688] A new study created in RDB with name: optimization_study


Creating a new study
Epoch 1/2000 - Loss: 0.6804 - Val: 0.6702 - Win: 0.6702
Epoch 2/2000 - Loss: 0.6660 - Val: 0.6633 - Win: 0.6667
Epoch 3/2000 - Loss: 0.6619 - Val: 0.6602 - Win: 0.6646
Epoch 4/2000 - Loss: 0.6609 - Val: 0.6625 - Win: 0.6641
Epoch 5/2000 - Loss: 0.6552 - Val: 0.6626 - Win: 0.6638


[I 2026-07-25 18:48:42,236] Trial 0 finished with value: 0.6602484376806962 and parameters: {'num_agg_layers': 6, 'agg_hidden_dim_1': 110, 'agg_hidden_dim_2': 170, 'agg_hidden_dim_3': 384, 'agg_hidden_dim_4': 408, 'agg_hidden_dim_5': 475, 'agg_hidden_dim_6': 413, 'num_lin_layers': 2, 'lin_hidden_dim_1': 200, 'lin_hidden_dim_2': 433, 'activation': 'elu', 'dropout_rate': 0.2792888479953828, 'optimizer': 'SGD', 'weight_decay': 4.3073037064069e-05}. Best is trial 0 with value: 0.6602484376806962.


Epoch 6/2000 - Loss: 0.6544 - Val: 0.6648 - Win: 0.6627
Early stopping
Restoring best model from epoch 3 with val_loss 0.6602
Epoch 1/2000 - Loss: 0.6755 - Val: 0.6959 - Win: 0.6959
Epoch 2/2000 - Loss: 0.6577 - Val: 0.7116 - Win: 0.7037
Epoch 3/2000 - Loss: 0.6631 - Val: 0.6606 - Win: 0.6894
Epoch 4/2000 - Loss: 0.6609 - Val: 0.6620 - Win: 0.6825
Epoch 5/2000 - Loss: 0.6524 - Val: 0.6527 - Win: 0.6766
Epoch 6/2000 - Loss: 0.6421 - Val: 0.6697 - Win: 0.6713
Epoch 7/2000 - Loss: 0.6290 - Val: 0.6671 - Win: 0.6624
Epoch 8/2000 - Loss: 0.6278 - Val: 0.7122 - Win: 0.6727
Epoch 9/2000 - Loss: 0.6320 - Val: 0.7059 - Win: 0.6815
Epoch 10/2000 - Loss: 0.6216 - Val: 0.6957 - Win: 0.6901


[I 2026-07-25 18:54:37,479] Trial 1 finished with value: 0.6526792501148425 and parameters: {'num_agg_layers': 3, 'agg_hidden_dim_1': 78, 'agg_hidden_dim_2': 319, 'agg_hidden_dim_3': 136, 'num_lin_layers': 3, 'lin_hidden_dim_1': 435, 'lin_hidden_dim_2': 200, 'lin_hidden_dim_3': 265, 'activation': 'leakyrelu', 'dropout_rate': 0.44321935953395153, 'optimizer': 'RAdam', 'weight_decay': 8.315300682620078e-06}. Best is trial 1 with value: 0.6526792501148425.


...

Epoch 12/2000 - Loss: 0.6375 - Val: 0.6311 - Win: 0.6304
Early stopping
Restoring best model from epoch 11 with val_loss 0.6241
Epoch 1/2000 - Loss: 0.6883 - Val: 0.6483 - Win: 0.6483
Epoch 2/2000 - Loss: 0.6803 - Val: 0.6891 - Win: 0.6687
Epoch 3/2000 - Loss: 0.6778 - Val: 0.6469 - Win: 0.6615
Epoch 4/2000 - Loss: 0.6751 - Val: 0.6596 - Win: 0.6610
Epoch 5/2000 - Loss: 0.6661 - Val: 0.6344 - Win: 0.6557


[I 2026-07-25 22:40:21,000] Trial 48 finished with value: 0.634421087566175 and parameters: {'num_agg_layers': 5, 'agg_hidden_dim_1': 336, 'agg_hidden_dim_2': 440, 'agg_hidden_dim_3': 280, 'agg_hidden_dim_4': 181, 'agg_hidden_dim_5': 147, 'num_lin_layers': 4, 'lin_hidden_dim_1': 51, 'lin_hidden_dim_2': 12, 'lin_hidden_dim_3': 31, 'lin_hidden_dim_4': 278, 'activation': 'selu', 'dropout_rate': 0.28697737492277364, 'optimizer': 'Adam', 'weight_decay': 7.092401196208413e-05}. Best is trial 18 with value: 0.6131702623869243.


Epoch 6/2000 - Loss: 0.6567 - Val: 0.6658 - Win: 0.6592
Early stopping
Restoring best model from epoch 5 with val_loss 0.6344
Epoch 1/2000 - Loss: 0.6769 - Val: 0.6532 - Win: 0.6532
Epoch 2/2000 - Loss: 0.6554 - Val: 0.6411 - Win: 0.6472
Epoch 3/2000 - Loss: 0.6457 - Val: 0.7563 - Win: 0.6836
Epoch 4/2000 - Loss: 0.6326 - Val: 0.7183 - Win: 0.6922
Epoch 5/2000 - Loss: 0.6245 - Val: 0.7718 - Win: 0.7081


[I 2026-07-25 22:43:32,079] Trial 49 finished with value: 0.6411411561464009 and parameters: {'num_agg_layers': 2, 'agg_hidden_dim_1': 34, 'agg_hidden_dim_2': 40, 'num_lin_layers': 3, 'lin_hidden_dim_1': 302, 'lin_hidden_dim_2': 159, 'lin_hidden_dim_3': 89, 'activation': 'selu', 'dropout_rate': 0.3307847734214368, 'optimizer': 'RMSprop', 'weight_decay': 4.2396221418143896e-05}. Best is trial 18 with value: 0.6131702623869243.


Epoch 6/2000 - Loss: 0.6207 - Val: 0.6849 - Win: 0.7145
Early stopping
Restoring best model from epoch 2 with val_loss 0.6411
Best hyperparameters: {'num_agg_layers': 5, 'agg_hidden_dim_1': 250, 'agg_hidden_dim_2': 338, 'agg_hidden_dim_3': 315, 'agg_hidden_dim_4': 87, 'agg_hidden_dim_5': 10, 'num_lin_layers': 4, 'lin_hidden_dim_1': 327, 'lin_hidden_dim_2': 162, 'lin_hidden_dim_3': 165, 'lin_hidden_dim_4': 22, 'activation': 'selu', 'dropout_rate': 0.3225334071987974, 'optimizer': 'Adam', 'weight_decay': 3.406889559239717e-05}


##### Run best model found in optuna optimization

In [10]:
from utils import device
from optimizer import retrain
from save import save_model, save_params

best_params = study.best_params
model, best_val_loss, min_val_loss, train_losses, val_losses = retrain(
    best_params,
    node_dim,
    edge_dim,
    train_loader,
    val_loader,
    num_tasks,
    architecture_type=architecture_type,
    lr=lr)

save_model(model, model_directory, filename)
save_params(best_params, params_directory, filename, device)

Epoch 1/2000 - Loss: 0.6686 - Val: 0.6559 - Win: 0.6559
Epoch 2/2000 - Loss: 0.6623 - Val: 0.6537 - Win: 0.6548
Epoch 3/2000 - Loss: 0.6519 - Val: 0.6553 - Win: 0.6549
Epoch 4/2000 - Loss: 0.6514 - Val: 0.6407 - Win: 0.6514
Epoch 5/2000 - Loss: 0.6528 - Val: 0.6351 - Win: 0.6481
Epoch 6/2000 - Loss: 0.6471 - Val: 0.6490 - Win: 0.6467
Epoch 7/2000 - Loss: 0.6465 - Val: 0.6305 - Win: 0.6421
Epoch 8/2000 - Loss: 0.6468 - Val: 0.6409 - Win: 0.6392
Epoch 9/2000 - Loss: 0.6368 - Val: 0.6376 - Win: 0.6386
Epoch 10/2000 - Loss: 0.6413 - Val: 0.6413 - Win: 0.6399
Epoch 11/2000 - Loss: 0.6422 - Val: 0.6341 - Win: 0.6369
Epoch 12/2000 - Loss: 0.6379 - Val: 0.6421 - Win: 0.6392
Epoch 13/2000 - Loss: 0.6388 - Val: 0.6454 - Win: 0.6401
Epoch 14/2000 - Loss: 0.6386 - Val: 0.6241 - Win: 0.6374
Epoch 15/2000 - Loss: 0.6364 - Val: 0.6497 - Win: 0.6391
Epoch 16/2000 - Loss: 0.6383 - Val: 0.6344 - Win: 0.6391
Epoch 17/2000 - Loss: 0.6335 - Val: 0.6174 - Win: 0.6342
Early stopping
Restoring best model from

##### Statistical analysis of best model

In [11]:
from utils import device
from statistical import ModelEvaluator

ModelEvaluator(
    model,
    device,
    train_loader,
    val_loader,
    test_loader,
    calibration=False)

c:\Users\vinic\anaconda3\envs\graph\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\vinic\anaconda3\envs\graph\lib\site-packages\sklearn\metrics\_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


ValueError: not enough values to unpack (expected 4, got 1)

##### Visualize predicted vs experimental values

In [ ]:
from logits import visualize_logits

visualize_logits(
    model,
    train_loader,
    val_loader,
    test_loader,
    task_index=0,
    out_path=fig_path1)

##### Visualize embeddings along the epochs

In [ ]:
from embeddings import visualize_embeddings

visualize_embeddings(
    in_path=emb_path,
    epoch=20,
    method=method,
    task_index=0,
    out_path=fig_path2)